# Affibody Interface Redesign for Improved VEGFR2 Binding

## Overview

This notebook walks through a complete pipeline for redesigning the binding interface of a
58-amino-acid affibody protein to improve its affinity against VEGFR2 (Vascular Endothelial
Growth Factor Receptor 2).

**Strategy:** Start from an experimentally determined (or computationally modelled)
affibody–VEGFR2 complex and use *partial diffusion* in RFdiffusion3 to explore mutations
at the binding interface while keeping the VEGFR2 structure fixed.

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Interface Redesign** | RFD3 (partial diffusion) | Redesign affibody interface residues while keeping VEGFR2 fixed |
| 2. **Sequence Optimisation** | MPNN | Design full sequences on the new backbone |
| 3. **Structure Validation** | RF3 | Re-fold to confirm designed sequences adopt the intended structure |
| 4. **Ranking** | Metrics | Score designs by pLDDT, ipTM, and interface RMSD |

### Partial Diffusion Concept

Rather than designing a binder from scratch, partial diffusion adds a small amount of noise
(`partial_t` Å) to the starting structure and then denoises it.  The result is a set of
structures that are similar to the starting complex but with optimised side-chain packing
and backbone geometry at the interface.  Lower `partial_t` → conservative (mostly side-chain
remodelling); higher `partial_t` → more backbone exploration.

```
Input complex  ──noise(partial_t)──▶  noised structure
                                              │
                              RFD3 denoise (VEGFR2 fixed)
                                              │
                                              ▼
                              redesigned affibody variants
```

---

## Prerequisites

- A PDB file of the affibody–VEGFR2 complex (chain A = affibody, chain B = VEGFR2)
- The foundry package installed (`pip install rc-foundry[all]`)
- RFD3, MPNN, and RF3 checkpoints downloaded


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# ── USER CONFIGURATION ────────────────────────────────────────────────────────
COMPLEX_PDB = '/mnt/home/woldring/foundry/examples/affibody_vegfr2/model.cif'

AFFIBODY_CHAIN = 'A'
VEGFR2_CHAIN   = 'B'

AFFIBODY_LEN = 58
VEGFR2_LEN   = 766

# Interface residues on the affibody that contact VEGFR2 (≤5 Å cutoff)
INTERFACE_RESIDUES = [6, 9, 10, 13, 14, 17, 24, 25, 27, 28, 31, 35, 36]

OUT_DIR = 'outputs/affibody_vegfr2'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Complex CIF : {COMPLEX_PDB}')
print(f'Interface residues on chain {AFFIBODY_CHAIN}: {INTERFACE_RESIDUES}')
print(f'Output dir  : {OUT_DIR}')

---

## Section 1 – Interface Redesign with RFD3 (Partial Diffusion)

We run three partial-diffusion experiments in parallel:

| Run | `partial_t` | Description |
|-----|-------------|-------------|
| Conservative | 5 Å | Side-chain remodelling, minimal backbone movement |
| Moderate | 10 Å | Moderate backbone relaxation around interface |
| Aggressive | 15 Å | Larger backbone rearrangements (highest diversity) |

For each run VEGFR2 coordinates are fully fixed; the affibody backbone is kept fixed
(`select_fixed_atoms`) but its interface side chains can be redesigned
(`select_unfixed_sequence`).

In [ ]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from rfd3.inference.input_parsing import DesignInputSpecification

seed_everything(42)

# Build the contig string: fix both chains, chain-break between them
contig = (
    f'{AFFIBODY_CHAIN}1-{AFFIBODY_LEN},/0,'
    f'{VEGFR2_CHAIN}1-{VEGFR2_LEN}'
)

# Build select_fixed_atoms:
#   – VEGFR2: fix everything (coordinates and sequence)
#   – Affibody: fix backbone only; side chains will be free to diffuse
select_fixed_atoms = {
    f'{VEGFR2_CHAIN}1-{VEGFR2_LEN}': 'ALL',
    f'{AFFIBODY_CHAIN}1-{AFFIBODY_LEN}': 'BKBN',
}

# Build select_unfixed_sequence:
#   Residues listed here will have their amino-acid identity redesigned by the model.
unfixed_seq = ','.join(f'{AFFIBODY_CHAIN}{r}' for r in INTERFACE_RESIDUES)

# VEGFR2 hotspot residues — chain B contact sites identified from the starting complex.
select_hotspots = "B131-135,B137,B164-165,B193-197,B213,B215-218,B253-257,B276,B286,B310-312"

# Common spec fields shared across all three runs
base_spec = dict(
    input=COMPLEX_PDB,
    contig=contig,
    select_fixed_atoms=select_fixed_atoms,
    select_unfixed_sequence=unfixed_seq,
    select_hotspots=select_hotspots,
    infer_ori_strategy='hotspots',
    plddt_enhanced=True,
    redesign_motif_sidechains=True,
)

# Three specs with increasing noise levels
specs = {
    'conservative': DesignInputSpecification(**base_spec, partial_t=5.0),
    'moderate':     DesignInputSpecification(**base_spec, partial_t=10.0),
    'aggressive':   DesignInputSpecification(**base_spec, partial_t=15.0),
}

print('Contig          :', contig)
print('Interface residues (unfixed seq):', unfixed_seq)
print('Hotspots        :', select_hotspots)
print('Design specs ready:', list(specs.keys()))

In [ ]:
# Run each partial-diffusion experiment.
# n_batches × diffusion_batch_size = total designs per noise level.
# With n_batches=2, diffusion_batch_size=8 → 16 designs per noise level (48 total).

N_BATCHES            = 2   # increase for production runs
DIFFUSION_BATCH_SIZE = 8

config = RFD3InferenceConfig(diffusion_batch_size=DIFFUSION_BATCH_SIZE)
engine = RFD3InferenceEngine(**config)

all_rfd3_outputs = {}
for name, spec in specs.items():
    print(f'\n── Running partial diffusion: {name} (partial_t={spec.partial_t} Å) ──')
    outputs = engine.run(
        inputs=spec,
        out_dir=f'{OUT_DIR}/rfd3_{name}',
        n_batches=N_BATCHES,
    )
    all_rfd3_outputs[name] = outputs
    n_designs = sum(len(v) for v in (outputs or {}).values())
    print(f'   Generated {n_designs} structures → {OUT_DIR}/rfd3_{name}/')

print('\nRFD3 partial diffusion complete.')

---

## Section 2 – Sequence Design with MPNN

For each RFD3-generated backbone we use LigandMPNN to design amino-acid sequences.

Key configuration choices:
- VEGFR2 chain sequence is **fixed** (only the affibody gets designed).
- We generate **10 sequences per backbone** for diversity.
- `temperature=0.1` gives near-optimal sequences; raise to ~0.3–0.5 for more diversity.

In [ ]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
import biotite.structure as struc
import numpy as np

mpnn_engine = MPNNInferenceEngine(
    model_type='ligand_mpnn',
    is_legacy_weights=True,
    out_directory=None,
    write_structures=False,
    write_fasta=False,
)

# Per-input options: fix VEGFR2 chain, design only the affibody chain
mpnn_input_config = {
    'batch_size':     10,     # sequences per backbone
    'temperature':    0.1,    # sampling temperature
    'chains_to_design': [AFFIBODY_CHAIN],  # only redesign affibody chain
    'remove_waters':  True,
}

all_mpnn_outputs = {}
for run_name, rfd3_out in all_rfd3_outputs.items():
    if rfd3_out is None:
        continue
    # Collect all atom arrays from this noise-level run
    backbones = [
        design.atom_array
        for batch in rfd3_out.values()
        for design in batch
    ]
    print(f'\nRunning MPNN on {len(backbones)} backbones from "{run_name}" ...')
    mpnn_out = mpnn_engine.run(
        input_dicts=[mpnn_input_config] * len(backbones),
        atom_arrays=backbones,
    )
    all_mpnn_outputs[run_name] = mpnn_out
    print(f'   → {len(mpnn_out)} designed sequences generated')

print('\nMPNN complete.')

---

## Section 3 – Structure Validation with RF3

Re-fold the top-ranked MPNN sequences with RF3 to verify the affibody adopts the
designed backbone and assess complex-level confidence (ipTM).

**Key metrics for ranking:**
| Metric | Target | Meaning |
|--------|--------|---------|
| `overall_plddt` | > 0.80 | Per-residue confidence |
| `iptm` | > 0.75 | Interface confidence (higher → better complex) |
| `ranking_score` | maximise | Overall model quality |

In [ ]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput

rf3_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)

all_rf3_outputs = {}
for run_name, mpnn_out in all_mpnn_outputs.items():
    print(f'\nRunning RF3 on "{run_name}" designs ...')
    rf3_inputs = [
        InferenceInput.from_atom_array(d.atom_array, example_id=f'{run_name}_{i}')
        for i, d in enumerate(mpnn_out)
    ]
    rf3_out = rf3_engine.run(
        inputs=rf3_inputs,
        annotate_b_factor_with_plddt=True,
    )
    all_rf3_outputs[run_name] = rf3_out
    print(f'   → {len(rf3_out)} predictions complete')

print('\nRF3 complete.')

---

## Section 4 – Ranking and Export

Score every design by pLDDT, ipTM, and backbone RMSD to the starting affibody,
then export the top-ranked structures for downstream analysis.

In [ ]:
import pandas as pd
import numpy as np
from biotite.structure import rmsd, superimpose
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
from atomworks.io.utils.io_utils import read_structure

# Load original affibody backbone for RMSD reference
original_complex  = read_structure(COMPLEX_PDB)
original_affibody = original_complex[original_complex.chain_id == AFFIBODY_CHAIN]
original_bb       = original_affibody[np.isin(original_affibody.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]

records = []
for run_name, rf3_out in all_rf3_outputs.items():
    for example_id, pred_list in rf3_out.items():
        pred = pred_list[0]  # top model
        sc   = pred.summary_confidences

        # Backbone RMSD of affibody vs starting structure (chain A only)
        pred_affibody = pred.atom_array[pred.atom_array.chain_id == AFFIBODY_CHAIN]
        pred_bb       = pred_affibody[np.isin(pred_affibody.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]
        try:
            pred_bb_fit, _ = superimpose(original_bb, pred_bb)
            bb_rmsd = float(rmsd(original_bb, pred_bb_fit))
        except Exception:
            bb_rmsd = float('nan')

        records.append({
            'run':            run_name,
            'example_id':     example_id,
            'plddt':          sc.get('overall_plddt', float('nan')),
            'iptm':           sc.get('iptm', float('nan')),
            'ptm':            sc.get('ptm', float('nan')),
            'ranking_score':  sc.get('ranking_score', float('nan')),
            'has_clash':      sc.get('has_clash', None),
            'bb_rmsd_A':      bb_rmsd,
        })

df = pd.DataFrame(records)

# Filter out clashing designs and sort by ipTM
df_clean = df[df['has_clash'] == False].sort_values('iptm', ascending=False)

print(f'Total designs: {len(df)}')
print(f'Non-clashing : {len(df_clean)}')
df_clean.head(20)

In [ ]:
from atomworks.io.utils.io_utils import to_cif_file

TOP_N = 10
export_dir = f'{OUT_DIR}/top_{TOP_N}'
os.makedirs(export_dir, exist_ok=True)

top_ids = df_clean.head(TOP_N)['example_id'].tolist()

for run_name, rf3_out in all_rf3_outputs.items():
    for example_id, pred_list in rf3_out.items():
        if example_id in top_ids:
            pred = pred_list[0]
            rank = top_ids.index(example_id) + 1
            out_path = f'{export_dir}/rank{rank:02d}_{example_id}'
            to_cif_file(pred.atom_array, out_path, file_type='cif')
            print(f'  Exported rank {rank}: {example_id}  (ipTM={df_clean[df_clean.example_id==example_id].iptm.values[0]:.3f})')

# Save full ranking table
df_clean.to_csv(f'{OUT_DIR}/design_ranking.csv', index=False)
print(f'\nRanking table saved to {OUT_DIR}/design_ranking.csv')

---

## Interpreting Results

### Metrics Guide

| Metric | Good threshold | Notes |
|--------|---------------|-------|
| `plddt` | ≥ 0.80 | Per-residue confidence; < 0.70 suggests poorly structured regions |
| `iptm` | ≥ 0.75 | Interface confidence; most reliable predictor of binding |
| `ranking_score` | maximise | Combined quality score |
| `bb_rmsd_A` | 0.5–3.0 Å | Low = conservative change; high = more remodelling |

### Recommended Next Steps

1. **Visual inspection** – Load top structures into PyMOL/ChimeraX and inspect the
   interface.  Confirm key interactions (H-bonds, hydrophobic contacts) are maintained
   or improved.

2. **Rosetta refinement** – Run FastRelax on the top designs to locally minimise energy.

3. **Experimental validation** – Prioritise designs with `iptm > 0.80`, no clashes, and
   conserved key contacts for expression and SPR/ITC binding assay.

4. **Iterative refinement** – Feed best experimental binders back into the pipeline and
   run another round with `partial_t=5.0` for fine-tuning.

### Command-line alternative

To run the RFD3 step directly from the CLI (using the provided YAML):

```bash
# Conservative run
rfd3 design \
  out_dir=outputs/affibody_vegfr2/rfd3_conservative \
  inputs=examples/affibody_vegfr2/inputs.yaml \
  json_keys_subset=affibody_vegfr2_conservative \
  n_batches=5 \
  diffusion_batch_size=16

# Moderate run
rfd3 design \
  out_dir=outputs/affibody_vegfr2/rfd3_moderate \
  inputs=examples/affibody_vegfr2/inputs.yaml \
  json_keys_subset=affibody_vegfr2_moderate \
  n_batches=5 \
  diffusion_batch_size=16
```
